# 02 · The grouped split and the leakage check

**ITRI626 · Colile Sibanda**

Objective **O2**, and the project's central invariant: a source document
belongs to exactly one split. A leaked split gives high test scores that
mean nothing, and nothing downstream would complain — so the check is
executed here with its output visible.


In [ ]:
# Every notebook opens with the same four lines: path, config, seed, paths.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from docclf.config.loader import load_config
from docclf.config.paths import resolve_paths
from docclf.config.seed import set_global_seed
from docclf.utils.env import detect_environment

CONFIG = ROOT / "configs" / "base.yaml"
ENV = detect_environment()
cfg = load_config(CONFIG, env=ENV)
paths = resolve_paths(cfg, ENV).ensure()
set_global_seed(cfg.seed)

f"Config: {cfg.name} | Environment: {ENV} | Seed: {cfg.seed}"

In [ ]:
import pandas as pd

working = pd.read_csv(paths.processed / 'working_set.csv')
print(f"Working-set pages: {len(working)}")
print(f"Source documents: {working['doc_group_id'].nunique()}")
print(f"Seed: {cfg.seed}")
print(f"Ratios (train/val/test): {cfg.split.ratios}")

## Draw the split

Whole documents are placed, largest first, into the split with the largest remaining deficit — per class, so the class balance follows the ratios as closely as whole groups allow.


In [ ]:
from docclf.splits.grouped import make_grouped_split

splits = make_grouped_split(
    working, cfg.split.ratios, cfg.seed, cfg.split.group_column, cfg.split.stratify_column
)
splits['split'].value_counts(normalize=True).round(4).to_frame('share')

## The leakage assertion

This raises if any `doc_group_id` appears under more than one split label.


In [ ]:
from docclf.splits.leakage import assert_no_group_overlap, check_near_duplicates

assert_no_group_overlap(splits, cfg.split.group_column)
print('Group overlap check: passed')

violations = check_near_duplicates(splits, max_distance=0)
print(f'Near-duplicate violations across splits: {len(violations)}')
for item in violations[:5]:
    print(item)

## What the check does when the split is poisoned

Evidence that the assertion actually fires, rather than passing vacuously.


In [ ]:
from docclf.splits.leakage import LeakageError

poisoned = splits.copy()
victim = poisoned.loc[poisoned['split'] == 'train', 'doc_group_id'].iloc[0]
poisoned.loc[poisoned.index[poisoned['doc_group_id'] == victim][0], 'split'] = 'test'
try:
    assert_no_group_overlap(poisoned, cfg.split.group_column)
    print('NOT DETECTED — this would invalidate the project')
except LeakageError as error:
    print(f'Rejected as expected: {error}')

## Per-split, per-class distribution

This is the table the report must contain. Its counts total the working set exactly.


In [ ]:
from docclf.splits.report import describe_splits, split_totals

summary = describe_splits(splits, cfg.split.group_column)
summary.to_csv(paths.splits / 'split_summary.csv', index=False)
assert int(summary['count'].sum()) == len(working)
split_totals(splits)

In [ ]:
pivot = summary.pivot(index='class_name', columns='split', values='count')
pivot = pivot.reindex(cfg.classes)[['train', 'val', 'test']]
ax = pivot.plot.bar(stacked=True, figsize=(9, 4))
ax.set_ylabel('Pages')
ax.set_title(f'Split composition per class (seed {cfg.seed})')
pivot

## Commit the split files

These CSVs are committed, so a reader recreates the exact experiment rather than a similar one.


In [ ]:
from docclf.splits.io import save_splits

save_splits(splits, paths.splits)